# Periodic Steady State

Finding a limit cycle directly, instead of integrating until the transient has died away.

## The problem with waiting

A driven oscillator settles onto a periodic orbit, but only asymptotically. "Settled" is measured below and lands around fifty periods for the system here — every one of them integrated at the accuracy the final answer needs, purely to be thrown away.

The steady state is a fixed point of the **period map**

$$g(x_0) = x(T;\,x_0),$$

the state after integrating exactly one period from $x_0$. Solving $g(x_0) = x_0$ finds the orbit without passing through the transient at all.

FastSim solves it by Anderson-accelerated shooting: each outer iteration integrates one period and takes an Anderson step on $(x_{\text{start}}, x_{\text{end}})$. Only function evaluations — no monodromy matrix $\partial x(T)/\partial x(0)$ to assemble or factorise.

## The System

The driven Duffing oscillator — a mass on a spring whose stiffness grows with deflection:

$$\ddot{x} + \delta\dot{x} + \alpha x + \beta x^3 = \gamma\cos(\omega t)$$

The cubic term is what makes it interesting: the response is not a scaled copy of the drive, and the orbit is not an ellipse.

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

# Apply the FastSim docs matplotlib style
plt.style.use('../fastsim_docs.mplstyle')

import fastsim as fs
from fastsim import Simulation, Connection
from fastsim.blocks import SinusoidalSource, DynamicalSystem, Scope
from fastsim.solvers import RKCK54

## System Parameters

These put the system on a period-1 orbit. The Duffing has parameter ranges with several coexisting attractors and chaotic ones besides; shooting finds *a* fixed point of the period map, so it is worth checking that the one it finds is the one the system settles onto — which is what the verification below does.

In [ ]:
alpha = 1.0       # linear stiffness
beta = 1.0        # cubic stiffness (hardening)
delta = 0.2       # damping
gamma = 1.5       # forcing amplitude
omega = 1.4       # forcing angular frequency

period = 2 * np.pi / omega
print(f"forcing period: {period:.4f} s")

## Block Diagram

A `SinusoidalSource` drives a `DynamicalSystem` carrying $[x, \dot{x}]$.

In [ ]:
SAMPLES_PER_PERIOD = 200

def build():
    # Record on a uniform grid rather than at the adaptive solver's step points,
    # so periods can be compared index for index without interpolating.
    src = SinusoidalSource(frequency=1.0 / period, amplitude=gamma, phase=0.0)
    duffing = DynamicalSystem(
        func_dyn=lambda x, u, t: np.array([
            x[1],
            -delta * x[1] - alpha * x[0] - beta * x[0] ** 3 + u[0],
        ]),
        func_alg=lambda x, u, t: x,
        initial_value=[1.0, 0.0],
    )
    sco = Scope(labels=["x", "v"], sampling_period=period / SAMPLES_PER_PERIOD)
    sim = Simulation(
        blocks=[src, duffing, sco],
        connections=[
            Connection(src, duffing),
            Connection(duffing[0], sco[0]),
            Connection(duffing[1], sco[1]),
        ],
        Solver=RKCK54,
        tolerance_lte_abs=1e-10,
        tolerance_lte_rel=1e-8,
        log=False,
    )
    return sim, sco

## Solving for the Limit Cycle

`periodic_steady_state` takes the period and does the rest. It leaves one period of the converged orbit in the `Scope`.

In [ ]:
sim, sco = build()

t0 = time.perf_counter()
sim.periodic_steady_state(period=period, reset=True)
t_pss = time.perf_counter() - t0

t_lc, [x_lc, v_lc] = sco.read()
print(f"converged in {t_pss * 1e3:.0f} ms")
print(f"one period recorded: {len(t_lc)} samples over {t_lc[-1]:.4f} s")

## Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
ax1.plot(t_lc, x_lc, label="x")
ax1.plot(t_lc, v_lc, label="v")
ax1.set_xlabel("time [s]")
ax1.legend()
ax2.plot(x_lc, v_lc)
ax2.set_xlabel("x")
ax2.set_ylabel("v")
plt.show()

The orbit is closed and visibly not elliptical — the cubic stiffness flattens it where the deflection is largest.

## Verification

Two things have to hold, and neither involves trusting the shooting method.

**It is periodic.** The state at the end of the recorded period must equal the state at its start. That is the defining property of the answer.

In [ ]:
start = np.array([x_lc[0], v_lc[0]])
end = np.array([x_lc[-1], v_lc[-1]])
print(f"x(0), v(0) = {start}")
print(f"x(T), v(T) = {end}")
print(f"closure residual ||x(T) - x(0)|| = {np.linalg.norm(end - start):.3e}")

**It is the orbit the system actually settles onto.** Integrating the transient long enough must arrive at the same cycle — that is the answer the shooting method is a shortcut to. How long "long enough" is, is itself worth measuring.

In [ ]:
# How many periods before the transient repeats itself? Compare each period to
# the one before it.
sim_s, sco_s = build()
sim_s.run(120 * period, reset=True)
t_s, [x_s, v_s] = sco_s.read()

print(f"{'periods':>9}{'||state_n - state_{n-1}||':>28}")
for n in (1, 5, 10, 20, 50, 100):
    i1, i0 = n * SAMPLES_PER_PERIOD, (n - 1) * SAMPLES_PER_PERIOD
    if i1 >= len(x_s):
        break
    print(f"{n:>9}{np.hypot(x_s[i1] - x_s[i0], v_s[i1] - v_s[i0]):>28.3e}")

In [ ]:
N_SETTLE = 100

sim_t, sco_t = build()

t0 = time.perf_counter()
sim_t.run(N_SETTLE * period, reset=True)
t_transient = time.perf_counter() - t0

t_tr, [x_tr, v_tr] = sco_t.read()
print(f"transient run: {N_SETTLE} periods in {t_transient * 1e3:.0f} ms")

In [ ]:
# Both runs sample the same uniform grid, so the last period of the transient
# lines up with the limit cycle sample for sample — no interpolation.
n = min(len(x_lc), SAMPLES_PER_PERIOD + 1)
x_tail, v_tail = x_tr[-n:], v_tr[-n:]

print(f"worst |x_pss - x_transient| = {np.max(np.abs(x_lc[:n] - x_tail)):.3e}")
print(f"worst |v_pss - v_transient| = {np.max(np.abs(v_lc[:n] - v_tail)):.3e}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(x_tr, v_tr, lw=0.6, alpha=0.5, label=f"transient, {N_SETTLE} periods")
ax.plot(x_lc, v_lc, lw=2.5, label="periodic_steady_state")
ax.set_xlabel("x")
ax.set_ylabel("v")
ax.legend()
plt.show()

The transient spirals in from its initial condition; the shooting solution is the orbit it spirals onto, reached without integrating the spiral.

In [ ]:
print(f"periodic_steady_state : {t_pss * 1e3:7.0f} ms")
print(f"{N_SETTLE}-period transient  : {t_transient * 1e3:7.0f} ms")